In [ ]:
%pip install numpy astropy astroquery matplotlib requests reproject

In [ ]:
# Funciones auxiliares (antes estaban en spherex_core.py; ahora van dentro del notebook)
"""SPHEREx Motion Explorer: comparación exploratoria de dos recortes FITS.

Basado en el tratamiento WCS del tutorial de IRSA/SPHEREx:
https://caltech-ipac.github.io/irsa-tutorials/spherex-cutouts/

Este módulo es una demostración visual, no un detector validado de variabilidad.
"""
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.visualization import ImageNormalize, AsinhStretch
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans
import astropy.units as u
from reproject import reproject_interp


def leer_recorte(ruta, ra, dec):
    """Lee un FITS y calcula la banda espectral en (RA, Dec), en grados."""
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs")

    with fits.open(ruta) as hdul:
        hdu = hdul["IMAGE"]
        imagen = np.asarray(hdu.data, dtype=float).copy()
        wcs = WCS(hdu.header).celestial
        x, y = wcs.world_to_pixel(coord)
        if not (0 <= x < imagen.shape[1] and 0 <= y < imagen.shape[0]):
            raise ValueError(f"{ruta.name}: las coordenadas no caen en el recorte.")

        # WCS-WAVE es alternativo y no debe aplicar la distorsión SIP espacial.
        wcs_espectral = WCS(hdu.header, fobj=hdul, key="W")
        wcs_espectral.sip = None
        longitud_onda, ancho_banda = wcs_espectral.pixel_to_world(x, y)
        metadatos = {
            "archivo": ruta.name,
            "lambda_um": float(longitud_onda.to_value(u.um)),
            "ancho_um": float(ancho_banda.to_value(u.um)),
            "x": float(x),
            "y": float(y),
            "unidad": str(hdu.header.get("BUNIT", "sin especificar")),
            "version": str(hdul[0].header.get("VERSION", "no indicada")),
        }
        # Conservar información para la validación de calidad posterior.
        flags = np.array(hdul["FLAGS"].data, copy=True)
        varianza = np.array(hdul["VARIANCE"].data, copy=True)
    return imagen, wcs, flags, varianza, metadatos


def rellenar_huecos(imagen, x, y, radio=6, iteraciones=5):
    """Interpola los NaN cercanos a (x, y) (pixeles). Devuelve (imagen, huecos).

    Solo toca NaN dentro de `radio` pixeles de la posicion indicada; el resto
    de la imagen queda igual. Los valores interpolados NO son flujo medido:
    sirven para visualizar y centrar, no para fotometria.
    """
    salida = imagen.copy()
    yy, xx = np.indices(imagen.shape)
    zona = np.hypot(xx - x, yy - y) <= radio
    huecos = zona & ~np.isfinite(imagen)
    if not huecos.any():
        return salida, huecos
    kernel = Gaussian2DKernel(1.5)
    for _ in range(iteraciones):
        tmp = interpolate_replace_nans(salida, kernel, boundary="extend")
        salida[huecos] = tmp[huecos]
        if not (huecos & ~np.isfinite(salida)).any():
            break
    return salida, huecos


def crear_blink(ruta_1, ruta_2, ra, dec, salida="05_Analisis/M101_blink_automatico.gif",
                etiquetas=None, rellenar_estrella=True, radio_relleno=6):
    """Alinea dos recortes FITS y genera un GIF de inspección visual.

    etiquetas: lista opcional de 2 textos (p. ej. las fechas) para el título
    de cada cuadro. Si es None, se muestra "Época N · λ = ... μm".

    rellenar_estrella: interpola los pixeles NaN alrededor de (ra, dec) antes
    de alinear (evita el hueco central de una estrella brillante).

    NO iguala las PSF, respuesta espectral, fondos ni calibraciones relativas.
    """
    if etiquetas is not None and len(etiquetas) != 2:
        raise ValueError("etiquetas debe tener exactamente 2 textos.")
    im1, wcs1, flags1, var1, meta1 = leer_recorte(ruta_1, ra, dec)
    im2, wcs2, flags2, var2, meta2 = leer_recorte(ruta_2, ra, dec)

    if meta1["unidad"] != meta2["unidad"]:
        raise ValueError("Las imágenes tienen unidades BUNIT distintas; revisa la calibración.")

    n_rell = (0, 0)
    if rellenar_estrella:
        im1, h1 = rellenar_huecos(im1, meta1["x"], meta1["y"], radio_relleno)
        im2, h2 = rellenar_huecos(im2, meta2["x"], meta2["y"], radio_relleno)
        n_rell = (int(h1.sum()), int(h2.sum()))

    alineada2, huella = reproject_interp(
        (im2, wcs2), wcs1, shape_out=im1.shape, order="bilinear"
    )
    comunes = (np.isfinite(im1) & np.isfinite(alineada2) & (huella > 0.99))
    if comunes.sum() < 10:
        raise ValueError("No hay suficientes píxeles válidos en la cobertura común.")

    # Ambas épocas usan el mismo rango visual, sin normalizar sus intensidades por separado.
    intensidades = np.concatenate([im1[comunes], alineada2[comunes]])
    vmin, vmax = np.percentile(intensidades, [1, 99.5])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        raise ValueError("No se pudo establecer una escala de intensidad válida.")
    norma = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())
    mapa = plt.colormaps["magma"].copy()
    mapa.set_bad("black")
    cuadros = [
        np.ma.masked_where(~comunes, im1),
        np.ma.masked_where(~comunes, alineada2),
    ]

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection=wcs1)
    vista = ax.imshow(
        cuadros[0], origin="lower", cmap=mapa, norm=norma,
        interpolation="nearest"
    )
    ax.set_xlabel("Ascensión recta (RA)")
    ax.set_ylabel("Declinación (Dec)")
    titulo = ax.set_title(etiquetas[0] if etiquetas else "Época 1")

    def actualizar(frame):
        vista.set_data(cuadros[frame])
        if etiquetas:
            titulo.set_text(etiquetas[frame])
        else:
            titulo.set_text(f"Época {frame + 1} · λ = { [meta1, meta2][frame]['lambda_um']:.4f} μm")
        return vista, titulo

    animacion = FuncAnimation(fig, actualizar, frames=2, interval=1000, blit=False)
    salida = Path(salida)
    salida.parent.mkdir(parents=True, exist_ok=True)
    try:
        animacion.save(str(salida), writer=PillowWriter(fps=1), dpi=130)
    finally:
        plt.close(fig)

    diferencia_lam = abs(meta1["lambda_um"] - meta2["lambda_um"])
    informe = {
        "archivo_gif": str(salida),
        "cobertura_comun_pct": float(comunes.mean() * 100),
        "delta_lambda_um": float(diferencia_lam),
        "lambda_1_um": meta1["lambda_um"],
        "lambda_2_um": meta2["lambda_um"],
        "ancho_1_um": meta1["ancho_um"],
        "ancho_2_um": meta2["ancho_um"],
        "unidad": meta1["unidad"],
        "pix_rellenados_1": n_rell[0],
        "pix_rellenados_2": n_rell[1],
    }
    return informe


def crear_blink_multiple(rutas, posiciones, salida="05_Analisis/blink_multiple.gif",
                         etiquetas=None, rellenar_estrella=True, radio_relleno=6,
                         fps=2):
    """Alinea N recortes FITS a la primera imagen y genera un GIF de inspeccion visual.

    rutas: lista de FITS, en orden temporal.
    posiciones: lista de (ra, dec) en grados; la posicion del objeto en la
        fecha de CADA observacion (importante para objetos con movimiento propio).
    etiquetas: lista opcional de N textos (uno por cuadro).

    Todas las imagenes usan la misma escala de intensidad. NO iguala las PSF,
    respuesta espectral, fondos ni calibraciones relativas: si las
    longitudes de onda difieren, el brillo puede cambiar por eso.
    """
    n = len(rutas)
    if n < 2:
        raise ValueError("Se necesitan al menos 2 imagenes.")
    if len(posiciones) != n:
        raise ValueError("rutas y posiciones deben tener la misma longitud.")
    if etiquetas is not None and len(etiquetas) != n:
        raise ValueError(f"etiquetas debe tener {n} textos.")

    datos = [leer_recorte(r, ra, dec) for r, (ra, dec) in zip(rutas, posiciones)]
    if len({d[4]["unidad"] for d in datos}) > 1:
        raise ValueError("Las imagenes tienen unidades BUNIT distintas; revisa la calibracion.")

    # Rellenar NaN junto al objeto ANTES de alinear (reproject los propaga).
    imagenes, n_rell = [], []
    for im, _wcs, _fl, _var, meta in datos:
        if rellenar_estrella:
            im, h = rellenar_huecos(im, meta["x"], meta["y"], radio_relleno)
            n_rell.append(int(h.sum()))
        else:
            n_rell.append(0)
        imagenes.append(im)

    # Alinear todas a la cuadricula de la primera.
    wcs_ref = datos[0][1]
    forma = imagenes[0].shape
    alineadas = [imagenes[0]]
    comunes = np.isfinite(imagenes[0])
    for im, d in zip(imagenes[1:], datos[1:]):
        al, huella = reproject_interp(
            (im, d[1]), wcs_ref, shape_out=forma, order="bilinear"
        )
        alineadas.append(al)
        comunes &= np.isfinite(al) & (huella > 0.99)
    if comunes.sum() < 10:
        raise ValueError("No hay suficientes pixeles validos en la cobertura comun.")

    intensidades = np.concatenate([a[comunes] for a in alineadas])
    vmin, vmax = np.percentile(intensidades, [1, 99.5])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        raise ValueError("No se pudo establecer una escala de intensidad valida.")
    norma = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())
    mapa = plt.colormaps["magma"].copy()
    mapa.set_bad("black")
    cuadros = [np.ma.masked_where(~comunes, a) for a in alineadas]

    # Posicion esperada del objeto en cada fecha, en la cuadricula comun.
    pix = []
    for ra, dec in posiciones:
        x, y = wcs_ref.world_to_pixel(SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs"))
        pix.append((float(x), float(y)))

    def texto(frame):
        if etiquetas:
            return etiquetas[frame]
        return f"Epoca {frame + 1} - lambda = {datos[frame][4]['lambda_um']:.4f} um"

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection=wcs_ref)
    vista = ax.imshow(cuadros[0], origin="lower", cmap=mapa, norm=norma,
                      interpolation="nearest")
    marcador = ax.scatter([pix[0][0]], [pix[0][1]], marker="o", facecolors="none",
                          edgecolors="cyan", s=220, linewidths=1.2)
    ax.set_xlabel("Ascension recta (RA)")
    ax.set_ylabel("Declinacion (Dec)")
    titulo = ax.set_title(texto(0))

    def actualizar(frame):
        vista.set_data(cuadros[frame])
        marcador.set_offsets([list(pix[frame])])
        titulo.set_text(texto(frame))
        return vista, marcador, titulo

    animacion = FuncAnimation(fig, actualizar, frames=n,
                              interval=int(1000 / fps), blit=False)
    salida = Path(salida)
    salida.parent.mkdir(parents=True, exist_ok=True)
    try:
        animacion.save(str(salida), writer=PillowWriter(fps=fps), dpi=130)
    finally:
        plt.close(fig)

    lambdas = [d[4]["lambda_um"] for d in datos]
    return {
        "archivo_gif": str(salida),
        "n_imagenes": n,
        "cobertura_comun_pct": float(comunes.mean() * 100),
        "lambdas_um": lambdas,
        "rango_lambda_um": float(max(lambdas) - min(lambdas)),
        "unidad": datos[0][4]["unidad"],
        "pix_rellenados": n_rell,
    }


In [ ]:
from pathlib import Path
import numpy as np
from astroquery.ipac.irsa import Irsa
from astropy.coordinates import SkyCoord, Distance
from astropy.time import Time
import astropy.units as u

OBJETO = "Barnard"

# Estrella de Barnard (Gaia DR3, ICRS, epoca J2000).
# Tiene el mayor movimiento propio conocido (~10.4"/ano), por eso su
# posicion se calcula para la fecha exacta de cada observacion.
estrella = SkyCoord(
    ra=269.452075 * u.deg,
    dec=4.693391 * u.deg,
    pm_ra_cosdec=-802.803 * u.mas / u.yr,
    pm_dec=10362.394 * u.mas / u.yr,
    distance=Distance(parallax=546.976 * u.mas),
    radial_velocity=-110.6 * u.km / u.s,
    frame="icrs",
    obstime=Time("J2000"),
)

def posicion_en(tiempo):
    """Posicion ICRS (sin movimiento propio adjunto) en la fecha dada."""
    p = estrella.apply_space_motion(new_obstime=tiempo)
    return SkyCoord(p.ra, p.dec, frame="icrs")

# Posicion aproximada durante la mision SPHEREx
centro = posicion_en(Time("2026-01-01"))
print(f"Posicion en 2026: RA={centro.ra.deg:.5f}  Dec={centro.dec.deg:.5f}")

# Buscar imagenes que cubran la posicion (radio amplio por el movimiento propio)
resultados = Irsa.query_sia(
    pos=(centro, 30 * u.arcsec),
    collection=["spherex_dr1", "spherex_qr3"]
)

# Filtrar por detector D2
d2 = resultados[
    resultados["energy_bandpassname"] == "SPHEREx-D2"
]

d2.sort("t_min")

# Crear carpeta y guardar los metadatos
carpeta = Path("03_Catalogos")
carpeta.mkdir(exist_ok=True)

for coleccion in ["spherex_dr1", "spherex_qr3"]:
    tabla = d2[d2["obs_collection"] == coleccion]
    tabla.write(
        carpeta / f"{OBJETO}_{coleccion}_D2.csv",
        format="ascii.csv",
        overwrite=True
    )
    print(f"{coleccion}: {len(tabla)} imagenes")

print("\nTotal D2:", len(d2))

In [ ]:
import io
import requests
from astropy.io import fits

# Cuantas imagenes descargar, repartidas a lo largo del tiempo
N_IMAGENES = 6

if len(d2) < 2:
    raise ValueError("Se necesitan al menos 2 observaciones D2 para esta posicion.")

# Elegir la observacion mas cercana a cada una de N fechas equiespaciadas
# (entre la primera y la ultima disponible, DR1 + QR3).
t_todas = np.asarray(d2["t_min"], dtype=float)
objetivos = np.linspace(t_todas.min(), t_todas.max(), min(N_IMAGENES, len(d2)))
indices = sorted({int(np.argmin(np.abs(t_todas - t))) for t in objetivos})
ids = [str(d2["obs_id"][i]) for i in indices]

# Para elegirlas a mano, descomenta y usa obs_id de los CSV guardados:
# ids = ["2025W50_1A_0005_3", "2026W23_2A_0336_3", ...]

print(f"Observaciones elegidas ({len(ids)}):", ids)

base = Path("04_Datos_FITS")
base.mkdir(exist_ok=True)

obs = []   # informacion de cada observacion descargada

for obs_id in ids:
    filas = d2[d2["obs_id"] == obs_id]

    if len(filas) != 1:
        print(f"No se encontro una fila unica para {obs_id}")
        continue

    fila = filas[0]
    coleccion = str(fila["obs_collection"])
    subcarpeta = base / ("DR1" if coleccion == "spherex_dr1" else "QR3")
    subcarpeta.mkdir(exist_ok=True)

    # Posicion de la estrella en la fecha de ESTA observacion
    t_obs = Time(float(fila["t_min"]), format="mjd")
    pos = posicion_en(t_obs)

    # URL del FITS original y peticion de recorte centrado en la estrella
    url = str(fila["access_url"])
    url_cutout = f"{url}?center={pos.ra.deg},{pos.dec.deg}&size=0.1"

    archivo = subcarpeta / f"{obs_id}_D2_cutout.fits"

    try:
        respuesta = requests.get(url_cutout, timeout=(20, 120))
        respuesta.raise_for_status()

        # Comprobar que el contenido sea FITS
        with fits.open(io.BytesIO(respuesta.content)) as hdul:
            hdul.verify("exception")
            print(obs_id, "Extensiones:", len(hdul))

        archivo.write_bytes(respuesta.content)
        print("Guardado:", archivo)

        fecha = t_obs.strftime("%Y-%m-%d")
        obs.append({
            "obs_id": obs_id,
            "ruta": archivo,
            "pos": pos,
            "t_obs": t_obs,
            "fecha": fecha,
            "titulo": f"{coleccion.replace('spherex_', '').upper()} - {fecha}",
        })

    except Exception as error:
        print("Error al descargar:", obs_id, error)

obs.sort(key=lambda o: o["t_obs"].mjd)   # orden temporal
print(f"\nDescargadas: {len(obs)} de {len(ids)}")

if len(obs) < 2:
    raise RuntimeError("Se necesitan al menos 2 imagenes descargadas para comparar.")

In [ ]:
from pathlib import Path
from astropy.io import fits

for o in obs:
    archivo = o["ruta"]
    print(f"\nArchivo: {archivo.name}")

    if not archivo.exists():
        print("NO SE ENCONTRO EL ARCHIVO")
        continue

    print(f"Tamano: {archivo.stat().st_size / 1024**2:.2f} MB")

    try:
        with fits.open(archivo) as hdul:
            hdul.verify("exception")
            hdul.info()

            if "IMAGE" in hdul:
                print("Imagen cientifica encontrada")
                print("Dimensiones:", hdul["IMAGE"].data.shape)
            else:
                print("No se encontro la extension IMAGE")

    except Exception as e:
        print("Error al leer el FITS:", e)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import (
    ImageNormalize, PercentileInterval, AsinhStretch
)

# =============================================
# 1. LEER CADA RECORTE CON leer_recorte()
#    (usa la posicion de la estrella EN LA FECHA de cada observacion)
# =============================================
datos = []
for o in obs:
    imagen, wcs, flags, varianza, meta = leer_recorte(
        o["ruta"], o["pos"].ra.deg, o["pos"].dec.deg
    )
    datos.append({"imagen": imagen, "wcs": wcs, "flags": flags,
                  "varianza": varianza, "meta": meta})

    print(f"\n{o['titulo']}")
    print(f"Longitud de onda: {meta['lambda_um']:.4f} um")
    print(f"Ancho de banda: {meta['ancho_um']:.4f} um")
    print(f"Posicion pixel: ({meta['x']:.2f}, {meta['y']:.2f})")
    print(f"Unidad: {meta['unidad']} | Version: {meta['version']}")

sep = obs[0]["pos"].separation(obs[-1]["pos"]).arcsec
lams = [d["meta"]["lambda_um"] for d in datos]
print(f"\nDesplazamiento esperado entre la primera y la ultima fecha: {sep:.2f} arcsec "
      f"(1 pixel SPHEREx ~ 6.2 arcsec)")
print(f"Rango de longitudes de onda: {min(lams):.3f} - {max(lams):.3f} um")

# =============================================
# 2. COMPARACION INICIAL (cada imagen en su propio WCS)
# =============================================
n = len(obs)
ncols = min(n, 3)
nrows = int(np.ceil(n / ncols))
fig = plt.figure(figsize=(4.5 * ncols, 4.5 * nrows))

for i, (o, d) in enumerate(zip(obs, datos), start=1):
    meta = d["meta"]
    norm = ImageNormalize(
        d["imagen"],
        interval=PercentileInterval(99),
        stretch=AsinhStretch()
    )
    ax = fig.add_subplot(nrows, ncols, i, projection=d["wcs"])
    ax.imshow(d["imagen"], origin="lower", cmap="magma", norm=norm)
    ax.scatter(meta["x"], meta["y"], marker="+", color="cyan",
               s=130, linewidths=1.5)
    ax.set_title(f"{o['titulo']}\n{meta['lambda_um']:.3f} um", fontsize=9)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("Barnard_D2_comparacion_inicial.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Diagnostico: que pasa en los pixeles centrales de la estrella
R = 6   # radio de revision en pixeles

for o, d in zip(obs, datos):
    meta = d["meta"]
    img = d["imagen"]
    yy, xx = np.indices(img.shape)
    zona = np.hypot(xx - meta["x"], yy - meta["y"]) <= R

    print(f"\n{o['titulo']}  (estrella en pixel {meta['x']:.1f}, {meta['y']:.1f})")
    print(f"  NaN en IMAGE dentro de {R} px: "
          f"{np.sum(zona & ~np.isfinite(img))} de {zona.sum()}")

    valores = np.unique(d["flags"][zona])
    bits = sorted({b for v in valores for b in range(32) if int(v) >> b & 1})
    print("  Bits activos en FLAGS dentro de la zona:", bits)

In [ ]:
from reproject import reproject_interp

n = len(obs)
wcs_ref = datos[0]["wcs"]

# ==========================================
# 1. RELLENAR HUECOS ALREDEDOR DE LA ESTRELLA (antes de alinear)
# ==========================================
imgs, huecos = [], []
for d in datos:
    im, h = rellenar_huecos(d["imagen"], d["meta"]["x"], d["meta"]["y"])
    imgs.append(im)
    huecos.append(h)
print("Pixeles rellenados por imagen:", [int(h.sum()) for h in huecos])

# ==========================================
# 2. ALINEAR TODAS A LA PRIMERA (por coordenadas del cielo)
#    Las estrellas fijas coinciden; Barnard se desplaza.
# ==========================================
forma = imgs[0].shape
alineadas = [imgs[0]]
huecos_al = [huecos[0]]
comunes = np.isfinite(imgs[0])

for im, h, d in zip(imgs[1:], huecos[1:], datos[1:]):
    al, huella = reproject_interp(
        (im, d["wcs"]), wcs_ref, shape_out=forma, order="bilinear"
    )
    hal, _ = reproject_interp(
        (h.astype(float), d["wcs"]), wcs_ref, shape_out=forma,
        order="nearest-neighbor"
    )
    alineadas.append(al)
    huecos_al.append(np.nan_to_num(hal) > 0.5)
    comunes &= np.isfinite(al) & (huella > 0.99)

if not np.any(comunes):
    raise ValueError("Las imagenes no tienen una region comun valida.")
print("Cobertura comun:", f"{100 * comunes.mean():.1f}%")

# ==========================================
# 3. POSICION DE LA ESTRELLA EN CADA IMAGEN (centroide)
# ==========================================
pix = [wcs_ref.world_to_pixel(o["pos"]) for o in obs]

def centroide(img, x0, y0, r=4, iteraciones=3):
    """Centroide ponderado por flujo; la ventana se recentra en cada paso."""
    yy, xx = np.indices(img.shape)
    fondo = np.nanmedian(img[np.isfinite(img)])
    for _ in range(iteraciones):
        m = (np.hypot(xx - x0, yy - y0) <= r) & np.isfinite(img)
        if not m.any():
            return np.nan, np.nan
        w = np.clip(img[m] - fondo, 0, None)
        if w.sum() == 0:
            return np.nan, np.nan
        x0 = (xx[m] * w).sum() / w.sum()
        y0 = (yy[m] * w).sum() / w.sum()
    return x0, y0

cents = [centroide(im, float(px), float(py))
         for im, (px, py) in zip(alineadas, pix)]
ok = [i for i, c in enumerate(cents) if np.all(np.isfinite(c))]
print(f"Centroides medidos: {len(ok)} de {n}")

# ==========================================
# 4. MOVIMIENTO MEDIDO VS ESPERADO
# ==========================================
if len(ok) >= 2:
    ref = obs[0]["pos"]

    def offsets(c):
        dra = (c.ra.deg - ref.ra.deg) * np.cos(np.radians(ref.dec.deg)) * 3600
        ddec = (c.dec.deg - ref.dec.deg) * 3600
        return dra, ddec

    medidos = np.array([offsets(wcs_ref.pixel_to_world(*cents[i])) for i in ok])
    esperados = np.array([offsets(obs[i]["pos"]) for i in ok])
    t = np.array([obs[i]["t_obs"].jyear for i in ok])

    if len(ok) >= 3 and np.ptp(t) > 0:
        pm_ra, _ = np.polyfit(t - t[0], medidos[:, 0], 1)
        pm_dec, _ = np.polyfit(t - t[0], medidos[:, 1], 1)
        print(f"\nMovimiento propio medido (exploratorio):"
              f"  RA*cos(dec) = {pm_ra * 1000:.0f} mas/ano,"
              f"  Dec = {pm_dec * 1000:.0f} mas/ano")
        print("Catalogo (Gaia DR3):              "
              "RA*cos(dec) = -803 mas/ano,  Dec = 10362 mas/ano")

    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
    for ax, k, nombre in zip(axes2, (0, 1), ("RA*cos(dec)", "Dec")):
        ax.plot(t, esperados[:, k], "-", color="gray", label="esperado (catalogo)")
        ax.plot(t, medidos[:, k], "o", color="tab:orange", label="medido (centroide)")
        ax.set_xlabel("Fecha (ano)")
        ax.set_ylabel(f"Desplazamiento en {nombre} (arcsec)")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig("Barnard_movimiento_medido.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("No hay suficientes centroides para medir el movimiento.")

# ==========================================
# 5. COMPARACION EN LA MISMA ESCALA
# ==========================================
valores = np.concatenate([a[comunes] for a in alineadas])
vmin, vmax = np.percentile(valores, [1, 99.5])
norm = ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch())

ncols = min(n, 3)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(
    nrows, ncols, figsize=(4.5 * ncols, 4.5 * nrows),
    subplot_kw={"projection": wcs_ref}, squeeze=False
)
axes = axes.ravel()

for ax, al, o, (px, py), hueco in zip(axes, alineadas, obs, pix, huecos_al):
    im = ax.imshow(np.where(comunes, al, np.nan), origin="lower",
                   cmap="magma", norm=norm)
    ax.scatter(px, py, marker="o", facecolors="none",
               edgecolors="cyan", s=220, linewidths=1.2)
    if hueco.any():   # contorno de los pixeles interpolados
        ax.contour(hueco.astype(float), levels=[0.5],
                   colors="white", linewidths=0.8)
    ax.set_title(f"Barnard | {o['titulo']}", fontsize=9)
    ax.set_xlabel("RA")
    ax.set_ylabel("Dec")

for ax in axes[n:]:
    ax.axis("off")

fig.colorbar(im, ax=list(axes[:n]), shrink=0.8,
             label="Intensidad (unidades del FITS)")
plt.savefig("Barnard_comparacion_alineada.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import display, Image

# Cada imagen se alinea a la primera usando la posicion de la estrella
# en la fecha de ESA observacion.
informe = crear_blink_multiple(
    [o["ruta"] for o in obs],
    [(o["pos"].ra.deg, o["pos"].dec.deg) for o in obs],
    salida="05_Analisis/Barnard_blink.gif",
    etiquetas=[
        f"Estrella de Barnard · {o['fecha']} · {d['meta']['lambda_um']:.2f} um"
        for o, d in zip(obs, datos)
    ],
    fps=2,
)

for clave, valor in informe.items():
    print(f"{clave}: {valor}")

if any(informe["pix_rellenados"]):
    print("\nNota: se interpolaron pixeles NaN junto a la estrella; esos "
          "valores no son flujo medido.")

if informe["rango_lambda_um"] > 0.05:
    print("\nAviso: las imagenes estan en longitudes de onda distintas; "
          "el cambio de brillo puede ser espectral, no solo movimiento.")

display(Image(filename=informe["archivo_gif"]))